In [1]:
# Clone repo
!git clone https://github.com/BorgwardtLab/SAT.git /kaggle/working/SAT

# PyTorch 2.x Patch
file_path_models = '/kaggle/working/SAT/sat/models.py'
with open(file_path_models, 'r') as file:
    code_models = file.read()
if "batch_first" not in code_models:
    code_models = code_models.replace("self.encoder = GraphTransformerEncoder(encoder_layer, num_layers)", 
                                      "encoder_layer.self_attn.batch_first = False\n        self.encoder = GraphTransformerEncoder(encoder_layer, num_layers)")
    with open(file_path_models, 'w') as file:
        file.write(code_models)

# GAT Extension Patch
file_path_gnn = '/kaggle/working/SAT/sat/gnn_layers.py'
with open(file_path_gnn, 'r') as f:
    code_gnn = f.read()
if "'gat'" not in code_gnn:
    code_gnn = code_gnn.replace("'rwgnn', 'khopgnn'", "'rwgnn', 'khopgnn', 'gat'")
gat_code = """    elif gnn_type == "gat":
        return gnn.GATConv(embed_dim, embed_dim // 8, heads=8)
    elif gnn_type == "gin":"""
if 'elif gnn_type == "gat":' not in code_gnn:
    code_gnn = code_gnn.replace('    elif gnn_type == "gin":', gat_code)
with open(file_path_gnn, 'w') as f:
    f.write(code_gnn)

Cloning into '/kaggle/working/SAT'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 91 (delta 38), reused 71 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 2.63 MiB | 29.61 MiB/s, done.
Resolving deltas: 100% (38/38), done.


In [2]:
!pip install torch_geometric ogb performer-pytorch einops -q
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.10.0+cu128.html -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 93.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 103.2 MB/s eta 0:00:00


In [3]:
import sys
if '/kaggle/working/SAT' not in sys.path:
    sys.path.insert(0, '/kaggle/working/SAT')

!cd /kaggle/working/SAT && PYTHONPATH=/kaggle/working/SAT python experiments/train_zinc.py \
    --seed 0 \
    --gnn-type gat \
    --k-hop 3 \
    --se gnn \
    --abs-pe rw \
    --abs-pe-dim 20 \
    --epochs 2000 \
    --batch-size 128 \
    --lr 0.001 \
    --weight-decay 1e-5 \
    --dropout 0.2 \
    --num-layers 6 \
    --num-heads 8 \
    --dim-hidden 64 \
    --outdir /kaggle/working/results_gat

Namespace(seed=0, dataset='ZINC', num_heads=8, num_layers=6, dim_hidden=64, dropout=0.2, epochs=2000, lr=0.001, weight_decay=1e-05, batch_size=128, abs_pe='rw', abs_pe_dim=20, outdir='/kaggle/working/results_gat/ZINC/seed0/rw_20/gat_3_0.2_0.001_1e-05_6_8_64_BN', warmup=5000, layer_norm=False, use_edge_attr=False, edge_dim=32, gnn_type='gat', k_hop=3, global_pool='mean', se='gnn', use_cuda=True, batch_norm=True, save_logs=True)
Extracting ../datasets/ZINC/molecules.zip
Processing...
Processing test dataset: 100%|████████████| 1000/1000 [00:00<00:00, 8886.93it/s]
Done!
/kaggle/working/SAT/experiments/train_zinc.py:207: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_dset, batch_size=args.batch_size,
Data(x=[29], edge_index=[2, 64], edge_attr=[64], y=[1, 1], complete_edge_index=[2, 841], degree=[29])
/kaggle/working/SAT/experiments/train_zinc.py:216: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instea